# Data Engineer Challenge

## Descripcion
Este notebook documenta la solucion al desafio de ingenieria de datos. Se procesan ~398MB de tweets para resolver 3 problemas, cada uno con **2 enfoques**:

- **`*_time`**: Optimizado por **tiempo de ejecucion** (orjson + lectura bulk en RAM)
- **`*_memory`**: Optimizado por **uso de memoria** (streaming linea a linea, solo Counters en RAM)

### Supuestos
1. El archivo es JSON Lines (~117K tweets, un JSON por linea)
2. Para emojis (Q2) se usa la libreria `emoji` que detecta emojis Unicode estandar
3. Para menciones (Q3) se usa regex `@(\w+)` sobre el campo `content` (mas eficiente que parsear el campo nested `mentionedUsers`)
4. Cada aparicion de un emoji/mencion cuenta individualmente

### Herramientas de profiling
- **Memoria**: `memory_profiler`
- **Tiempo**: `cProfile` (Python Profilers)

## Setup

In [1]:
import sys
import os
import cProfile
import pstats
import io
from memory_profiler import memory_usage

# Notebook lives in src/, add current directory to path
sys.path.insert(0, os.getcwd())

from logger_config import get_logger
logger = get_logger("challenge")

# Data file path (one level above src/)
file_path = os.path.join(os.getcwd(), "..", "farmers-protest-tweets-2021-2-4.json")

# Verify file exists
file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
logger.info(f"File: {file_path}")
logger.info(f"Size: {file_size_mb:.1f} MB")

2026-03-04 20:50:54 [929189651.py] - INFO - File: c:\Users\Marlon Viasus\Downloads\challenge_data_ops_eng\src\..\farmers-protest-tweets-2021-2-4.json
2026-03-04 20:50:54 [929189651.py] - INFO - Size: 388.8 MB


In [2]:
# Mutable container to capture result and profiler stats inside memory_usage (which executes the function in a single pass)
_profile_data = {}


def _profiled_call(func, file_path):
    """Wrapper that runs func with cProfile and stores result + stats."""
    profiler = cProfile.Profile()
    profiler.enable()
    result = func(file_path)
    profiler.disable()

    stats_stream = io.StringIO()
    stats = pstats.Stats(profiler, stream=stats_stream)
    stats.sort_stats("cumulative")

    _profile_data["result"] = result
    _profile_data["total_time"] = stats.total_tt
    _profile_data["stats"] = stats


def measure(func, file_path, label=""):
    """Run a function ONCE measuring time (cProfile) + memory (memory_profiler)."""
    _profile_data.clear()

    # memory_usage runs _profiled_call, which in turn uses cProfile. This way the function runs ONCE and we capture both metrics
    peak_mem = memory_usage(
        (_profiled_call, (func, file_path)),
        max_usage=True,
    )
    peak_mb = peak_mem if isinstance(peak_mem, (int, float)) else max(peak_mem)

    result = _profile_data["result"]
    total_time = _profile_data["total_time"]

    logger.info(f"{label}: time={total_time:.2f}s | peak_mem={peak_mb:.1f} MiB")
    return result, total_time, peak_mb


def show_profile(func, file_path, label="", top_n=10):
    """Show cProfile detail for a function (top N calls)."""
    profiler = cProfile.Profile()
    profiler.enable()
    func(file_path)
    profiler.disable()

    stats_stream = io.StringIO()
    stats = pstats.Stats(profiler, stream=stats_stream)
    stats.sort_stats("cumulative")
    stats.print_stats(top_n)

    logger.info(f"cProfile detail for {label}:")
    for line in stats_stream.getvalue().split("\n"):
        if line.strip():
            logger.info(line)

---
## Q1: Top 10 fechas con mas tweets

**Objetivo**: Encontrar las 10 fechas con mas tweets y el usuario mas activo en cada una de esas fechas.

**Retorno**: `List[Tuple[datetime.date, str]]`

### Enfoque Time
- Lee todo el archivo de una vez con `readlines()` (I/O bulk)
- Parsea cada linea con `orjson` (parser en Rust, ~5x mas rapido que json stdlib)
- **Ventaja**: I/O rapido + parser rapido = menor tiempo total
- **Desventaja**: Requiere cargar todo el archivo en RAM

### Enfoque Memory
- Lee el archivo linea a linea con `open()` + `json.loads()`
- Usa `Counter` para contar tweets por fecha y `defaultdict(Counter)` para (fecha, usuario)
- Extrae la fecha directamente como substring `date[:10]` (evita parsear datetime completo)
- **Ventaja**: Solo contadores en RAM (~KB vs ~GB)
- **Desventaja**: Lectura secuencial mas lenta

In [3]:
from q1_time import q1_time
from q1_memory import q1_memory

r1_time, t1_time, m1_time = measure(q1_time, file_path, "q1_time")
r1_memory, t1_memory, m1_memory = measure(q1_memory, file_path, "q1_memory")

2026-03-04 20:50:56 [q1_time.py] - INFO - Starting q1_time -- bulk read + orjson
2026-03-04 20:50:57 [q1_time.py] - INFO - Read 117407 lines in bulk
2026-03-04 20:50:59 [q1_time.py] - INFO - q1_time completed -- 10 results
2026-03-04 20:50:59 [4049838931.py] - INFO - q1_time: time=2.96s | peak_mem=494.3 MiB
2026-03-04 20:51:01 [q1_memory.py] - INFO - Starting q1_memory -- streaming read
2026-03-04 20:51:08 [q1_memory.py] - INFO - Processed 117407 tweets via streaming
2026-03-04 20:51:08 [q1_memory.py] - INFO - q1_memory completed -- 10 results
2026-03-04 20:51:08 [4049838931.py] - INFO - q1_memory: time=6.97s | peak_mem=86.3 MiB


In [4]:
logger.info("Q1 - Top 10 dates with most tweets:")
for i, (date, user) in enumerate(r1_memory, 1):
    logger.info(f"{i:<4} {str(date):<15} {user}")

logger.info(f"[CHECK] Results match: {set(r1_time) == set(r1_memory)}")

r1_memory

2026-03-04 20:51:08 [3966585380.py] - INFO - Q1 - Top 10 dates with most tweets:
2026-03-04 20:51:08 [3966585380.py] - INFO - 1    2021-02-12      RanbirS00614606
2026-03-04 20:51:08 [3966585380.py] - INFO - 2    2021-02-13      MaanDee08215437
2026-03-04 20:51:08 [3966585380.py] - INFO - 3    2021-02-17      RaaJVinderkaur
2026-03-04 20:51:08 [3966585380.py] - INFO - 4    2021-02-16      jot__b
2026-03-04 20:51:08 [3966585380.py] - INFO - 5    2021-02-14      rebelpacifist
2026-03-04 20:51:08 [3966585380.py] - INFO - 6    2021-02-18      neetuanjle_nitu
2026-03-04 20:51:08 [3966585380.py] - INFO - 7    2021-02-15      jot__b
2026-03-04 20:51:08 [3966585380.py] - INFO - 8    2021-02-20      MangalJ23056160
2026-03-04 20:51:08 [3966585380.py] - INFO - 9    2021-02-23      Surrypuria
2026-03-04 20:51:08 [3966585380.py] - INFO - 10   2021-02-19      Preetm91
2026-03-04 20:51:08 [3966585380.py] - INFO - [CHECK] Results match: True


[(datetime.date(2021, 2, 12), 'RanbirS00614606'),
 (datetime.date(2021, 2, 13), 'MaanDee08215437'),
 (datetime.date(2021, 2, 17), 'RaaJVinderkaur'),
 (datetime.date(2021, 2, 16), 'jot__b'),
 (datetime.date(2021, 2, 14), 'rebelpacifist'),
 (datetime.date(2021, 2, 18), 'neetuanjle_nitu'),
 (datetime.date(2021, 2, 15), 'jot__b'),
 (datetime.date(2021, 2, 20), 'MangalJ23056160'),
 (datetime.date(2021, 2, 23), 'Surrypuria'),
 (datetime.date(2021, 2, 19), 'Preetm91')]

---
## Q2: Top 10 emojis mas usados

**Objetivo**: Encontrar los 10 emojis mas usados en todos los tweets con su conteo.

**Retorno**: `List[Tuple[str, int]]`

### Enfoque Time
- Lectura bulk + `orjson` para extraer emojis con `emoji.emoji_list()`
- **Ventaja**: Parser rapido
- **Desventaja**: Alto uso de RAM

### Enfoque Memory
- Streaming linea a linea, extrae emojis de cada tweet inmediatamente
- Solo un `Counter` en memoria
- **Ventaja**: Minimo uso de RAM
- **Desventaja**: Lectura secuencial es mas lenta

In [5]:
from q2_time import q2_time
from q2_memory import q2_memory

r2_time, t2_time, m2_time = measure(q2_time, file_path, "q2_time")
r2_memory, t2_memory, m2_memory = measure(q2_memory, file_path, "q2_memory")

2026-03-04 20:51:09 [q2_time.py] - INFO - Starting q2_time -- bulk read + orjson
2026-03-04 20:51:10 [q2_time.py] - INFO - Read 117407 lines in bulk
2026-03-04 20:51:57 [q2_time.py] - INFO - q2_time completed -- 870 unique emojis found
2026-03-04 20:51:57 [4049838931.py] - INFO - q2_time: time=48.23s | peak_mem=495.3 MiB
2026-03-04 20:51:58 [q2_memory.py] - INFO - Starting q2_memory -- streaming read
2026-03-04 20:52:50 [q2_memory.py] - INFO - Processed 117407 tweets -- 870 unique emojis
2026-03-04 20:52:51 [4049838931.py] - INFO - q2_memory: time=52.04s | peak_mem=86.4 MiB


In [6]:
logger.info("Q2 - Top 10 most used emojis:")
for i, (emj, count) in enumerate(r2_memory, 1):
    logger.info(f"{i:<4} {emj:<8} {count:,}")

logger.info(f"[CHECK] Results match: {set(r2_time) == set(r2_memory)}")

r2_memory

2026-03-04 20:52:51 [3496288957.py] - INFO - Q2 - Top 10 most used emojis:
2026-03-04 20:52:51 [3496288957.py] - INFO - 1    🙏        5,049
2026-03-04 20:52:51 [3496288957.py] - INFO - 2    😂        3,072
2026-03-04 20:52:51 [3496288957.py] - INFO - 3    🚜        2,972
2026-03-04 20:52:51 [3496288957.py] - INFO - 4    🌾        2,182
2026-03-04 20:52:51 [3496288957.py] - INFO - 5    🇮🇳       2,086
2026-03-04 20:52:51 [3496288957.py] - INFO - 6    🤣        1,668
2026-03-04 20:52:51 [3496288957.py] - INFO - 7    ✊        1,651
2026-03-04 20:52:51 [3496288957.py] - INFO - 8    ❤️       1,382
2026-03-04 20:52:51 [3496288957.py] - INFO - 9    🙏🏻       1,317
2026-03-04 20:52:51 [3496288957.py] - INFO - 10   💚        1,040
2026-03-04 20:52:51 [3496288957.py] - INFO - [CHECK] Results match: True


[('🙏', 5049),
 ('😂', 3072),
 ('🚜', 2972),
 ('🌾', 2182),
 ('🇮🇳', 2086),
 ('🤣', 1668),
 ('✊', 1651),
 ('❤️', 1382),
 ('🙏🏻', 1317),
 ('💚', 1040)]

---
## Q3: Top 10 usuarios mas mencionados

**Objetivo**: Encontrar los 10 usuarios mas mencionados (@username) en todos los tweets con su conteo.

**Retorno**: `List[Tuple[str, int]]`

### Enfoque Time
- Lectura bulk + `orjson` + regex pre-compilado
- **Ventaja**: Parser rapido + regex compilado
- **Desventaja**: Alto uso de RAM

### Enfoque Memory
- Streaming + regex sobre cada linea
- **Ventaja**: Solo un Counter en memoria
- **Desventaja**: Lectura secuencial

### Nota de diseno
Se decidio usar regex sobre `content` en lugar del campo `mentionedUsers` porque:
1. `mentionedUsers` es un objeto nested (requiere parseo adicional de objetos complejos)
2. Algunos tweets tienen `mentionedUsers: null`
3. La regex captura directamente las menciones visibles del tweet

In [7]:
from q3_time import q3_time
from q3_memory import q3_memory

r3_time, t3_time, m3_time = measure(q3_time, file_path, "q3_time")
r3_memory, t3_memory, m3_memory = measure(q3_memory, file_path, "q3_memory")

2026-03-04 20:52:52 [q3_time.py] - INFO - Starting q3_time -- bulk read + orjson
2026-03-04 20:52:52 [q3_time.py] - INFO - Read 117407 lines in bulk
2026-03-04 20:52:55 [q3_time.py] - INFO - q3_time completed -- 15675 unique users mentioned
2026-03-04 20:52:55 [4049838931.py] - INFO - q3_time: time=2.80s | peak_mem=496.0 MiB
2026-03-04 20:52:56 [q3_memory.py] - INFO - Starting q3_memory -- streaming read
2026-03-04 20:53:03 [q3_memory.py] - INFO - Processed 117407 tweets -- 15675 unique users
2026-03-04 20:53:03 [4049838931.py] - INFO - q3_memory: time=6.68s | peak_mem=73.4 MiB


In [8]:
logger.info("Q3 - Top 10 most mentioned users:")
for i, (user, count) in enumerate(r3_memory, 1):
    logger.info(f"{i:<4} {user:<20} {count:,}")

logger.info(f"[CHECK] Results match: {set(r3_time) == set(r3_memory)}")

r3_memory

2026-03-04 20:53:03 [3845327367.py] - INFO - Q3 - Top 10 most mentioned users:
2026-03-04 20:53:03 [3845327367.py] - INFO - 1    narendramodi         2,261
2026-03-04 20:53:03 [3845327367.py] - INFO - 2    Kisanektamorcha      1,836
2026-03-04 20:53:03 [3845327367.py] - INFO - 3    RakeshTikaitBKU      1,639
2026-03-04 20:53:03 [3845327367.py] - INFO - 4    PMOIndia             1,422
2026-03-04 20:53:03 [3845327367.py] - INFO - 5    RahulGandhi          1,125
2026-03-04 20:53:03 [3845327367.py] - INFO - 6    GretaThunberg        1,046
2026-03-04 20:53:03 [3845327367.py] - INFO - 7    RaviSinghKA          1,015
2026-03-04 20:53:03 [3845327367.py] - INFO - 8    rihanna              972
2026-03-04 20:53:03 [3845327367.py] - INFO - 9    UNHumanRights        962
2026-03-04 20:53:03 [3845327367.py] - INFO - 10   meenaharris          925
2026-03-04 20:53:03 [3845327367.py] - INFO - [CHECK] Results match: True


[('narendramodi', 2261),
 ('Kisanektamorcha', 1836),
 ('RakeshTikaitBKU', 1639),
 ('PMOIndia', 1422),
 ('RahulGandhi', 1125),
 ('GretaThunberg', 1046),
 ('RaviSinghKA', 1015),
 ('rihanna', 972),
 ('UNHumanRights', 962),
 ('meenaharris', 925)]

---
## Resumen comparativo

In [9]:
logger.info("COMPARATIVE SUMMARY")
logger.info(f"{'Function':<15} {'Time (s)':<12} {'Mem (MiB)':<12} {'Approach'}")
logger.info("-" * 55)
logger.info(f"{'q1_time':<15} {t1_time:<12.2f} {m1_time:<12.1f} {'orjson+bulk'}")
logger.info(f"{'q1_memory':<15} {t1_memory:<12.2f} {m1_memory:<12.1f} {'streaming'}")
logger.info(f"{'q2_time':<15} {t2_time:<12.2f} {m2_time:<12.1f} {'orjson+bulk'}")
logger.info(f"{'q2_memory':<15} {t2_memory:<12.2f} {m2_memory:<12.1f} {'streaming'}")
logger.info(f"{'q3_time':<15} {t3_time:<12.2f} {m3_time:<12.1f} {'orjson+bulk'}")
logger.info(f"{'q3_memory':<15} {t3_memory:<12.2f} {m3_memory:<12.1f} {'streaming'}")

{
    "q1_time": {"time_s": round(t1_time, 2), "mem_mib": round(m1_time, 1)},
    "q1_memory": {"time_s": round(t1_memory, 2), "mem_mib": round(m1_memory, 1)},
    "q2_time": {"time_s": round(t2_time, 2), "mem_mib": round(m2_time, 1)},
    "q2_memory": {"time_s": round(t2_memory, 2), "mem_mib": round(m2_memory, 1)},
    "q3_time": {"time_s": round(t3_time, 2), "mem_mib": round(m3_time, 1)},
    "q3_memory": {"time_s": round(t3_memory, 2), "mem_mib": round(m3_memory, 1)},
}

2026-03-04 20:53:03 [1080849280.py] - INFO - COMPARATIVE SUMMARY
2026-03-04 20:53:03 [1080849280.py] - INFO - Function        Time (s)     Mem (MiB)    Approach
2026-03-04 20:53:03 [1080849280.py] - INFO - -------------------------------------------------------
2026-03-04 20:53:03 [1080849280.py] - INFO - q1_time         2.96         494.3        orjson+bulk
2026-03-04 20:53:03 [1080849280.py] - INFO - q1_memory       6.97         86.3         streaming
2026-03-04 20:53:03 [1080849280.py] - INFO - q2_time         48.23        495.3        orjson+bulk
2026-03-04 20:53:03 [1080849280.py] - INFO - q2_memory       52.04        86.4         streaming
2026-03-04 20:53:03 [1080849280.py] - INFO - q3_time         2.80         496.0        orjson+bulk
2026-03-04 20:53:03 [1080849280.py] - INFO - q3_memory       6.68         73.4         streaming


{'q1_time': {'time_s': 2.96, 'mem_mib': 494.3},
 'q1_memory': {'time_s': 6.97, 'mem_mib': 86.3},
 'q2_time': {'time_s': 48.23, 'mem_mib': 495.3},
 'q2_memory': {'time_s': 52.04, 'mem_mib': 86.4},
 'q3_time': {'time_s': 2.8, 'mem_mib': 496.0},
 'q3_memory': {'time_s': 6.68, 'mem_mib': 73.4}}

---
## Detalle de profiling (cProfile)

In [10]:
show_profile(q1_time, file_path, "q1_time")
show_profile(q1_memory, file_path, "q1_memory")

2026-03-04 20:53:03 [q1_time.py] - INFO - Starting q1_time -- bulk read + orjson
2026-03-04 20:53:04 [q1_time.py] - INFO - Read 117407 lines in bulk
2026-03-04 20:53:05 [q1_time.py] - INFO - q1_time completed -- 10 results
2026-03-04 20:53:05 [4049838931.py] - INFO - cProfile detail for q1_time:
2026-03-04 20:53:05 [4049838931.py] - INFO -          290039 function calls (289913 primitive calls) in 2.510 seconds
2026-03-04 20:53:05 [4049838931.py] - INFO -    Ordered by: cumulative time
2026-03-04 20:53:05 [4049838931.py] - INFO -    List reduced from 251 to 10 due to restriction <10>
2026-03-04 20:53:05 [4049838931.py] - INFO -    ncalls  tottime  percall  cumtime  percall filename:lineno(function)
2026-03-04 20:53:05 [4049838931.py] - INFO -         1    0.335    0.335    1.475    1.475 c:\Users\Marlon Viasus\Downloads\challenge_data_ops_eng\src\q1_time.py:24(q1_time)
2026-03-04 20:53:05 [4049838931.py] - INFO -    117407    1.159    0.000    1.159    0.000 {orjson.loads}
2026-03-04 2

In [11]:
show_profile(q2_time, file_path, "q2_time")
show_profile(q2_memory, file_path, "q2_memory")

2026-03-04 20:53:11 [q2_time.py] - INFO - Starting q2_time -- bulk read + orjson
2026-03-04 20:53:12 [q2_time.py] - INFO - Read 117407 lines in bulk
2026-03-04 20:54:00 [q2_time.py] - INFO - q2_time completed -- 870 unique emojis found
2026-03-04 20:54:00 [4049838931.py] - INFO - cProfile detail for q2_time:
2026-03-04 20:54:00 [4049838931.py] - INFO -          85985724 function calls (85985536 primitive calls) in 48.869 seconds
2026-03-04 20:54:00 [4049838931.py] - INFO -    Ordered by: cumulative time
2026-03-04 20:54:00 [4049838931.py] - INFO -    List reduced from 256 to 10 due to restriction <10>
2026-03-04 20:54:00 [4049838931.py] - INFO -    ncalls  tottime  percall  cumtime  percall filename:lineno(function)
2026-03-04 20:54:00 [4049838931.py] - INFO -    117407    7.319    0.000   44.957    0.000 C:\Users\Marlon Viasus\AppData\Roaming\Python\Python313\site-packages\emoji\core.py:353(emoji_list)
2026-03-04 20:54:00 [4049838931.py] - INFO -  17140706   22.926    0.000   34.860  

In [12]:
show_profile(q3_time, file_path, "q3_time")
show_profile(q3_memory, file_path, "q3_memory")

2026-03-04 20:54:51 [q3_time.py] - INFO - Starting q3_time -- bulk read + orjson
2026-03-04 20:54:51 [q3_time.py] - INFO - Read 117407 lines in bulk
2026-03-04 20:54:53 [q3_time.py] - INFO - q3_time completed -- 15675 unique users mentioned
2026-03-04 20:54:53 [4049838931.py] - INFO - cProfile detail for q3_time:
2026-03-04 20:54:53 [4049838931.py] - INFO -          1059786 function calls (1059548 primitive calls) in 2.645 seconds
2026-03-04 20:54:53 [4049838931.py] - INFO -    Ordered by: cumulative time
2026-03-04 20:54:53 [4049838931.py] - INFO -    List reduced from 241 to 10 due to restriction <10>
2026-03-04 20:54:53 [4049838931.py] - INFO -    ncalls  tottime  percall  cumtime  percall filename:lineno(function)
2026-03-04 20:54:53 [4049838931.py] - INFO -         1    0.290    0.290    1.780    1.780 c:\Users\Marlon Viasus\Downloads\challenge_data_ops_eng\src\q3_time.py:28(q3_time)
2026-03-04 20:54:53 [4049838931.py] - INFO -    117407    1.154    0.000    1.154    0.000 {orjson

---
## Posibles mejoras

### Optimizacion de tiempo
- **Q2/Q3**: Paralelizar con `multiprocessing.Pool` para procesar chunks del archivo en paralelo
- **General**: Usar `pyarrow` para lectura columnar del JSON (solo leer columnas necesarias)

### Optimizacion de memoria
- **Q1**: Usar `ijson` para parsing SAX-style (aun menos memoria que json.loads por linea)
- **Q2**: Pre-compilar el set de emojis Unicode en vez de usar la libreria emoji por cada tweet
- **Q3**: Pre-compilar el patron regex con `re.compile()` fuera del loop